In [ ]:
!pip install -q peft==0.9.0 numpy==1.26.4 transformers==4.42.4 accelerate==0.33.0
!pip uninstall -y opencv-python-headless thinc spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 167.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 168.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# Generar PLS con CoT Phi-3.5-mini-instruct (QLoRA o bf16)
# Columnas esperadas: name, article, summary
# Entorno probado: transformers==4.42.4, accelerate==0.33.0, peft==0.9.0
# ============================================================
import re
import json, shutil, tempfile
import os
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel  # peft==0.9.0 recomendado

# --------- Parámetros generales ----------
DATA_DIR   = Path("/content/drive/MyDrive/MAIA-PROYECTO")
RESULTS_DIR= Path("/content/drive/MyDrive/MAIA-PROYECTO/models/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "microsoft/phi-3.5-mini-instruct"
LORA_DIR   = "/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora"  # ajusta si cambia
CKPT_DIR   = "/content/drive/MyDrive/MAIA-PROYECTO/outputs/phi3.5-mini-qlora/checkpoint-240"  # opcional


# Usa 4-bit si hay bitsandbytes con CUDA; si falla, pon False y carga en bf16
USE_4BIT   = True

# --------- Carga de datos ----------
df = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
for col in ["name", "article", "summary"]:
    assert col in df.columns, f"Falta la columna requerida: {col}"

# --------- Prompt coherente con tu entrenamiento ----------
def prompt_cot_factual(t):
    return f"""You are a helpful medical writer.
            Think briefly before answering:
            - Use only statements explicitly present in the source.
            - Keep names and numbers exactly as written.
            - 4–6 sentences, ≤120 words. Do not show your reasoning.

            Scientific text:
            {t}

            Plain summary:"""
# --------- Carga tokenizer ----------
print("[Modelo] Cargando tokenizer base…")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"   # más robusto en lotes para causal LM

# --------- Carga modelo base (4-bit o bf16) ----------
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
torch.backends.cuda.matmul.allow_tf32 = True

if USE_4BIT:
    try:
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16
        )
        print("[Modelo] Cargando modelo base en 4-bit…")
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            device_map="auto",
            quantization_config=bnb_cfg,
            torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
            attn_implementation="eager",   # evita flash-attn/triton
            trust_remote_code=True
        )
    except Exception as e:
        print(f"⚠️ 4-bit no disponible ({e}). Cargando en bf16…")
        USE_4BIT = False

if not USE_4BIT:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="auto",
        torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
        attn_implementation="eager",
        trust_remote_code=True
    )

# --------- Montar adaptador LoRA ----------
print("[Modelo] Montando adaptador LoRA fine-tuneado…")

def sanitize_lora_config(src_dir: Path) -> Path:
    """Crea una copia temporal del adaptador LoRA quitando claves no soportadas (p. ej. use_qalora)."""
    cfg_path = src_dir / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No se encontró {cfg_path}")
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    # Elimina claves nuevas que peft 0.9.0 no reconoce
    drop_keys = ["use_qalora", "rank_dropout", "module_dropout", "alpha_pattern",
                 "init_lora_weights", "fan_in_fan_out", "quant_storage_dtype"]
    for k in drop_keys:
        cfg.pop(k, None)

    tmp_dir = Path(tempfile.mkdtemp(prefix="lora_sanitized_"))
    shutil.copytree(src_dir, tmp_dir, dirs_exist_ok=True)
    with open(tmp_dir / "adapter_config.json", "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)
    return tmp_dir

try:
    sanitized_lora = sanitize_lora_config(Path(LORA_DIR))
    model = PeftModel.from_pretrained(base_model, str(sanitized_lora))
    if CKPT_DIR and os.path.isdir(CKPT_DIR):
        try:
            model.load_adapter(CKPT_DIR, adapter_name="resume")
            model.set_adapter("resume")
        except Exception as e:
            print("ℹ️ No se aplicó checkpoint adicional:", e)
    print("✅ Adaptador LoRA cargado correctamente.")
except Exception as e:
    print(f"❌ Error cargando LoRA ({e}) → usando solo modelo base.")
    model = base_model

model.eval()
print(f"✅ Modelo listo en {next(model.parameters()).device}, dtype={next(model.parameters()).dtype}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

[Modelo] Cargando tokenizer base…


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


⚠️ 4-bit no disponible (No package metadata was found for bitsandbytes). Cargando en bf16…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[Modelo] Montando adaptador LoRA fine-tuneado…
✅ Adaptador LoRA cargado correctamente.
✅ Modelo listo en cuda:0, dtype=torch.bfloat16
PyTorch: 2.6.0+cu124 | device: cuda


In [ ]:

# --------- Parámetros de generación (rápidos y estables para PLS) ----------
BATCH_SIZE = 4          # L4 aguanta 4 cómodamente (sube a 6 si cabe VRAM)
MAX_NEW    = 160        # ~120 palabras
GEN_KW = dict(
    max_new_tokens=MAX_NEW,
    do_sample=False,                # determinista → más factual
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.02,
    use_cache=True,                 # ✅ activo con transformers 4.42.4
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# --------- Generador por lotes ----------
@torch.inference_mode()
def generate_batch(prompts):
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2000,    # evita contextos gigantes
    ).to(device)

    out_ids = model.generate(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        **GEN_KW
    )

    # Decodificar solo la parte generada (corta el prompt)
    input_lens = enc["attention_mask"].sum(dim=1)
    texts = []
    for i in range(out_ids.size(0)):
        gen_only = out_ids[i, int(input_lens[i].item()):]
        text = tokenizer.decode(gen_only, skip_special_tokens=True).strip()
        texts.append(text)
    return texts

# --------- Inferencia y guardado ----------
articulos = df["article"].fillna("").astype(str).tolist()
pls, tiempos = [], []

print(f"[Inferencia] Generando {len(articulos)} resúmenes…")
for i in tqdm(range(0, len(articulos), BATCH_SIZE), desc="Generando PLS", unit="batch"):
    batch = articulos[i:i+BATCH_SIZE]
    prompts = [prompt_cot_factual(t) for t in batch]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = torch.cuda.Event(enable_timing=True); t1 = torch.cuda.Event(enable_timing=True)
    t0.record()

    batch_out = generate_batch(prompts)

    t1.record()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        ms = t0.elapsed_time(t1)   # milisegundos GPU
        dur = ms/1000.0
    else:
        import time
        dur = 0.0  # si no hay cuda, ignora

    pls.extend(batch_out)
    tiempos.extend([dur] * len(batch))

df["pls_phi35"] = pls
df["latency_s"] = tiempos

csv_out = RESULTS_DIR / "summaries_phi35_pls.csv"
df.to_csv(csv_out, index=False, encoding="utf-8")
print(f"✅ Guardado CSV con resúmenes en: {csv_out}")
print(f"⏱️ Latencia promedio (s): {df['latency_s'].mean():.2f}")

#20 a 22GB de VRAM

[Inferencia] Generando 380 resúmenes…


Generando PLS:   0%|          | 0/95 [00:00<?, ?batch/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


✅ Guardado CSV con resúmenes en: /content/drive/MyDrive/MAIA-PROYECTO/models/results/summaries_phi35_pls.csv
⏱️ Latencia promedio (s): 18.44


In [ ]:
import transformers
print(transformers.__version__)

4.53.2
